# Capítulo 2 · Populações, amostras e dados: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, a base de altas para casa de janeiro de um hospital de 500 leitos, pelas perguntas do gestor; aparecem só as seções com cálculo (2.7, 2.9 e 2.10). A **Parte B** traz a resolução escrita de todos os exercícios do capítulo (1 a 9; os enunciados estão no livro) e, nos exercícios cuja resolução precisa de cálculo (7, 8 e 9), o código e o resultado. O código é Python 3 simples, só com a biblioteca padrão. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/register.csv`: 1.200 linhas, uma por episódio de alta de janeiro de 2026, e cinco colunas: `episode_id`, `patient_id`, `formalized_at` (quando a alta foi formalizada), `departed_at` (quando o paciente saiu) e `delayed` (já calculada a partir dos dois horários; a primeira célula a calcula de novo). Cada célula traz o número da seção do livro a que pertence.

### 2.7 Definindo o evento a ser contado

*Quando uma alta conta como atrasada, e quantos dos 120 episódios sorteados foram?*

In [1]:
# PROGRAMA DE AUDITORIA: quanto esperaram os pacientes sorteados depois de a
# alta ser formalizada, e quantos esperaram mais de 2 horas?
#
# Pseudocódigo
#   1. Ler a base de altas de janeiro, um registro por episódio.
#   2. Para cada episódio, calcular a espera em horas, da alta formalizada
#      até a saída. Um horário faltante deixa a espera desconhecida.
#   3. Marcar o episódio como atrasado se a espera for MAIOR que 2 horas.
#   4. Sortear a amostra da auditoria: 120 episódios, ao acaso, sem reposição.
#   5. Relatar sobre a amostra: atrasados, porcentagem, mediana, maior, desconhecidos.
#   6. Contar de novo pela regra "2 horas ou mais", para ver o que
#      muda quando uma espera de exatamente 2 horas conta como atraso.

import csv
import random
import statistics
from datetime import datetime

TARGET_HOURS = 2  # a regra da auditoria: atrasado significa MAIS de 2 horas

# 1. Ler a base.
with open("data/register.csv", newline="") as stream:
    population = list(csv.DictReader(stream))

# 2-3. Calcular cada espera e marcar os atrasos.
for row in population:
    row["wait"] = None
    row["delayed"] = None
    if row["formalized_at"] and row["departed_at"]:
        formalized = datetime.fromisoformat(row["formalized_at"])
        departed = datetime.fromisoformat(row["departed_at"])
        wait = (departed - formalized).total_seconds() / 3600
        if wait >= 0:
            row["wait"] = wait
            row["delayed"] = wait > TARGET_HOURS
        else:
            print("Conferir horários:", row["episode_id"])

# 4. Sortear a amostra; a semente (11) torna o sorteio reproduzível.
sample = random.Random(11).sample(population, 120)

# 5. Relatar sobre a amostra.
waits = [row["wait"] for row in sample if row["wait"] is not None]
delayed = [wait for wait in waits if wait > TARGET_HOURS]
print("Episódios revisados:", len(sample))
print("Espera desconhecida:", len(sample) - len(waits))
print("Acima de 2 horas:", len(delayed), "de", len(waits))
print("Porcentagem acima de 2 horas:", 100 * len(delayed) / len(waits))
print("Espera mediana (horas):", statistics.median(waits))
print("Maior espera (horas):", max(waits))

# 6. As mesmas esperas pela regra "2 horas ou mais".
at_least_two = [wait for wait in waits if wait >= TARGET_HOURS]
share = 100 * len(at_least_two) / len(waits)
print("2 horas ou mais:", len(at_least_two), "de", len(waits))
print("Porcentagem de 2 horas ou mais:", share)

Episódios revisados: 120
Espera desconhecida: 0
Acima de 2 horas: 24 de 120
Porcentagem acima de 2 horas: 20.0
Espera mediana (horas): 1.5
Maior espera (horas): 8.0
2 horas ou mais: 45 de 120
Porcentagem de 2 horas ou mais: 37.5


### 2.9 Variação amostral

*Outro sorteio aleatório de 120 episódios daria a mesma proporção?*

In [2]:
# VARIAÇÃO AMOSTRAL: sorteios aleatórios diferentes dão a mesma proporção?
#
# Pseudocódigo
#   1. Contar os episódios atrasados na base inteira (só é possível
#      porque os dados são sintéticos; uma equipe de auditoria não a teria).
#   2. Sortear três amostras de 120 episódios, cada uma com uma semente diferente.
#   3. Em cada amostra, contar os episódios atrasados e a porcentagem.

def count_delayed(records):
    """Número de registros cuja situação de atraso é True (desconhecida não é contada)."""
    count = 0
    for row in records:
        if row["delayed"] is True:
            count = count + 1
    return count

# 1. A base inteira.
N = len(population)
K = count_delayed(population)
print(f"Base inteira: {K} atrasados de {N} ({100 * K / N:.2f}%)")

# 2-3. Três amostras; as sementes 11, 17 e 42 tornam os sorteios reproduzíveis.
sample_number = 0
for seed in [11, 17, 42]:
    sample_number = sample_number + 1
    draw = random.Random(seed).sample(population, 120)
    k = count_delayed(draw)
    n = len(draw)
    print(f"Amostra {sample_number}: {k} atrasados de {n} ({100 * k / n:.2f}%)")

Base inteira: 216 atrasados de 1200 (18.00%)
Amostra 1: 24 atrasados de 120 (20.00%)
Amostra 2: 22 atrasados de 120 (18.33%)
Amostra 3: 18 atrasados de 120 (15.00%)


### 2.10 Observações faltantes

*O que a equipe pode relatar quando falta o horário da alta formalizada em 12 dos 120 registros?*

In [3]:
# OBSERVAÇÕES FALTANTES: o que a equipe pode relatar quando faltam horários?
#
# Pseudocódigo
#   1. Fazer uma cópia separada dos 120 registros sorteados.
#   2. Na cópia, esconder a situação de atraso dos 12 primeiros episódios, como se
#      o horário da alta formalizada não tivesse sido registrado.
#   3. Contar as situações selecionadas, conhecidas e desconhecidas e os atrasos conhecidos.
#   4. Relatar a proporção de atrasos entre as situações conhecidas.
#   5. Relatar a faixa dos 120 episódios selecionados: toda situação desconhecida
#      no prazo (mínimo) ou toda situação desconhecida atrasada (máximo).

# 1-2. Copiar os registros (os originais ficam intactos) e esconder 12 situações.
incomplete = [dict(row) for row in sample]
for row in incomplete[:12]:
    row["delayed"] = None

# 3. Contagens.
selected = len(incomplete)
known = [row for row in incomplete if row["delayed"] is not None]
unknown = selected - len(known)
known_delays = count_delayed(known)

# 4-5. Resultados.
print("Selecionados:", selected)
print("Situação conhecida:", len(known))
print("Situação desconhecida:", unknown)
print("Atrasos conhecidos:", known_delays)
print(f"Atrasados entre os conhecidos: {100 * known_delays / len(known):.1f}%")
lowest = 100 * known_delays / selected
highest = 100 * (known_delays + unknown) / selected
print(f"Todos os selecionados: {lowest:.1f}% a {highest:.1f}%")

Selecionados: 120
Situação conhecida: 108
Situação desconhecida: 12
Atrasos conhecidos: 23
Atrasados entre os conhecidos: 21.3%
Todos os selecionados: 19.2% a 29.2%


## Parte B. Soluções dos exercícios

### 2.13 Exercício 1. Leitos, episódios e pacientes

A unidade é uma alta para casa, o episódio de alta para casa. A população-alvo
consiste nos 1.200 episódios elegíveis de alta neste hospital durante janeiro.
Os 500 leitos descrevem a capacidade; 1.080 é o número de pacientes distintos associados
a esses episódios. Nenhum dos dois é o número de episódios.

Um paciente pode contribuir com dois episódios elegíveis. Ambos pertencem a uma
auditoria baseada em episódios. Se a pergunta dissesse respeito, em vez disso, a pacientes com pelo menos uma alta atrasada,
a unidade passaria a ser o paciente, e a população-alvo consistiria nos 1.080 pacientes
distintos associados aos episódios elegíveis. Suas internações precisam ser vinculadas de modo que
cada pessoa seja contada uma vez e um atraso em qualquer internação elegível seja identificado. Contar
episódios não pode, por si só, fornecer a proporção no nível do paciente.

### 2.13 Exercício 2. Identificar o intervalo relevante

O intervalo entre a alta formalizada e a saída é de 2,5 horas, o que excede a referência de duas horas.
Os 30 minutos adicionais até o leito ficar disponível dizem respeito ao preparo
do leito, não à espera do paciente após a alta ser formalizada. O terceiro registro de data e hora permitiria
uma investigação separada de quando os leitos desocupados ficam disponíveis para novas admissões.

### 2.13 Exercício 3. Um marco amostral incompleto

A população-alvo contém todos os 1.200 episódios elegíveis. O marco contém apenas
os 900 episódios de dias úteis. A amostra contém os 300 episódios selecionados. Os 300
episódios de fim de semana não têm nenhuma chance de seleção a partir desse marco. Selecionar 600 em vez de
300 episódios de dias úteis ainda excluiria todos os episódios de fim de semana.

Solicite uma exportação corrigida com todos os dias elegíveis e uma entrada por episódio, concilie-a
com a base de 1.200 episódios e só então selecione de acordo com o plano documentado.
A cobertura do marco precisa ser corrigida antes que uma amostra maior possa responder à
pergunta original.

### 2.13 Exercício 4. Um censo tem uma fronteira definida

Se todas as 240 consultas elegíveis forem incluídas, a cobertura é um censo das consultas
de abril. Uma pessoa com três consultas agendadas contribui com três unidades para
esse censo. Uma pergunta no nível do paciente exige uma definição separada e a vinculação das
consultas pertencentes à mesma pessoa. A lista de consultas também não precisa incluir
todos os pacientes cadastrados na clínica.

A cobertura completa das consultas não verifica a exatidão dos registros de comparecimento.
Ela também não estabelece o padrão de comparecimento em maio, quando as consultas agendadas,
os pacientes e as condições do serviço podem ser diferentes. Cobertura, exatidão do registro
e aplicação a outro período são perguntas distintas.

### 2.13 Exercício 5. Um parâmetro e uma estatística para espécimes rejeitados

O parâmetro populacional é a proporção rejeitada entre todos os 800 espécimes elegíveis:

$$ p=\frac{40}{800}=0{,}05=5\%. $$

A estatística amostral é a proporção rejeitada entre os 80 espécimes selecionados:

$$ \widehat{p}=\frac{6}{80}=0{,}075=7{,}5\%. $$

A estatística amostral é 2,5 pontos percentuais maior. A amostra contém apenas uma
seleção de espécimes; outra seleção pode conter um número diferente de rejeitados.
A população continua com 800 espécimes e 40 rejeições. Nenhum intervalo de incerteza
nem recomendação de tamanho de amostra decorre apenas dessa comparação.

### 2.13 Exercício 6. Registros, duplicatas e observações faltantes

A unidade é uma visita de reabilitação. As entradas identificam dois pacientes, P01 e P02, e três visitas distintas, V01, V02
e V03. As duas visitas de P01 são unidades legítimas e separadas. V03 aparece duas vezes com a
mesma informação registrada. Verifique a fonte para confirmar se se trata de uma entrada
duplicada antes de manter um único registro para essa visita.

A duração de V03 é desconhecida. Zero minuto afirmaria uma duração conhecida e substituiria
a informação faltante por uma afirmação diferente. O serviço tem evidência de três
visitas, mas de informação de duração para apenas duas delas. Essas contagens devem permanecer
separadas.

### 2.13 Exercício 7. Três sorteios de uma população-alvo

O código deste exercício e do próximo está na Parte B do caderno complementar, que
executa antes as células do capítulo até a Seção 2.9: elas leem a base,
classificam os atrasos e criam as amostras usadas a seguir.

As Amostras 1, 2 e 3 contêm 24, 22 e 18 episódios atrasados,
respectivamente: 20,00%, 18,33% e 15,00%. A população-alvo permanece com 1.200 episódios,
incluindo 216 atrasos (18%). Cada sorteio seleciona 120 entradas distintas sem reposição.

Para verificar a unicidade, o caderno reúne os identificadores dos episódios em um `set`. Um conjunto
mantém cada valor distinto uma vez; comparar seu tamanho com o tamanho da amostra detecta
identificadores repetidos. Uma instrução `assert` verifica uma condição e interrompe a
execução se a condição for falsa; aqui ela verifica as afirmações numéricas impressas.

Todas as verificações passam: cada amostra contém 120 episódios distintos, e a base
continua com 1.200 episódios e 216 atrasos. As amostras contêm episódios
diferentes, de modo que suas proporções podem diferir. Três
amostras realizadas ilustram a variação; elas não caracterizam sua distribuição completa.

In [4]:
for seed in [11, 17, 42]:
    draw = random.Random(seed).sample(population, 120)
    identifiers = set()
    for row in draw:
        identifiers.add(row["episode_id"])
    assert len(draw) == len(identifiers) == 120
assert len(population) == 1200
assert K == 216

### 2.13 Exercício 8. Uma amostra incompleta

A amostra contém 120 episódios selecionados: 108 situações conhecidas, 12 situações desconhecidas
e 23 atrasos conhecidos. Entre as situações conhecidas, $23/108\approx21{,}3\%$. Entre todos os episódios
selecionados, o número de atrasados pode ser tão pequeno quanto 23 ou tão grande quanto $23+12=35$,
o que dá aproximadamente 19,2%-29,2%. Esta última faixa descreve o grupo selecionado
sob informação faltante; não é um intervalo de confiança populacional.

O caderno recria a Amostra 1, oculta 12 situações em uma cópia separada
e confirma a proporção observada e os dois limites calculados acima.

A saída do caderno confirma 23 atrasos conhecidos entre 108 situações observadas (21,3%).
Os limites usam todos os 120 episódios selecionados: atribuir as 12 situações desconhecidas
a pontual ou atrasado dá 19,2% ou 29,2%, respectivamente.

In [5]:
sample = random.Random(11).sample(population, 120)
incomplete = []
for row in sample:
    incomplete.append(dict(row))
for row in incomplete[:12]:
    row["delayed"] = None

known_n = 0
known_delays = 0
for row in incomplete:
    if row["delayed"] is not None:
        known_n = known_n + 1
        if row["delayed"] is True:
            known_delays = known_delays + 1
selected_n = len(incomplete)
missing_n = selected_n - known_n
print(selected_n, known_n, missing_n, known_delays)
# 120 108 12 23
print(f"Situações conhecidas: {100 * known_delays / known_n:.1f}%")
# Situações conhecidas: 21.3%
lower = 100 * known_delays / selected_n
upper = 100 * (known_delays + missing_n) / selected_n
print(f"Grupo selecionado: {lower:.1f}%-{upper:.1f}%")
# Grupo selecionado: 19.2%-29.2%

120 108 12 23
Situações conhecidas: 21.3%
Grupo selecionado: 19.2%-29.2%


### 2.13 Exercício 9. Definir a próxima auditoria de processo

Os intervalos posteriores à desocupação somam $20+50+30+10+40=150$ minutos. A espera pelo
início da limpeza é a mais longa, com 50 minutos; a limpeza em si leva 30 minutos.

Para um estudo de janeiro sobre essa espera, um evento de desocupação de leito é a unidade.
Defina as enfermarias e as datas elegíveis, liste todos os eventos de desocupação elegíveis e obtenha os
registros de data e hora de desocupação, de solicitação e de início da limpeza. Inclua os eventos que ainda aguardam
limpeza; restringir a base às limpezas concluídas excluiria as esperas não resolvidas.
A equipe também deve definir o momento de corte da observação e registrar a situação de cada evento
naquele momento. Revise os registros de atribuição de tarefas e de escala de pessoal para investigar por que uma espera ocorreu.
Um evento ilustrado identifica um intervalo a estudar, não a sua frequência em todo o hospital.

In [6]:
# 1. Os seis eventos de um episódio sintético de desocupação de leito (da Figura 2.2)
events = [
    ("leito desocupado", "2026-01-04T13:00"),
    ("limpeza solicitada", "2026-01-04T13:20"),
    ("limpeza iniciada", "2026-01-04T14:10"),
    ("limpeza concluída", "2026-01-04T14:40"),
    ("leito pronto", "2026-01-04T14:50"),
    ("próximo paciente entra", "2026-01-04T15:30"),
]

# 2. Minutos entre cada evento e o seguinte, com o maior intervalo
total = 0
longest_name = ""
longest_minutes = 0
for i in range(len(events) - 1):
    name_a, time_a = events[i]
    name_b, time_b = events[i + 1]
    minutes = (datetime.fromisoformat(time_b) - datetime.fromisoformat(time_a)).total_seconds() / 60
    print(name_a, "até", name_b + ":", minutes, "minutos")
    total = total + minutes
    if minutes > longest_minutes:
        longest_minutes = minutes
        longest_name = name_a + " até " + name_b
print("Total da desocupação do leito até a entrada do próximo paciente:", total, "minutos")
print("Maior intervalo:", longest_name, "(" + str(longest_minutes) + " minutos)")

leito desocupado até limpeza solicitada: 20.0 minutos
limpeza solicitada até limpeza iniciada: 50.0 minutos
limpeza iniciada até limpeza concluída: 30.0 minutos
limpeza concluída até leito pronto: 10.0 minutos
leito pronto até próximo paciente entra: 40.0 minutos
Total da desocupação do leito até a entrada do próximo paciente: 150.0 minutos
Maior intervalo: limpeza solicitada até limpeza iniciada (50.0 minutos)


---
*Dados sintéticos (`generate_data.py`, sem números aleatórios). Valores apenas didáticos; nada aqui é estimativa para um hospital real.*